# R7. Auto-optimise and hand in (15 min)
A **GEPA-style reflective loop**: show the model the mistakes your prompt makes on dev, ask it to diagnose *why*, accept a rewritten prompt, and keep the candidates that are best on **quality and cost** (a Pareto set).

Then you **submit one final prompt**. The instructor runs it on the hidden test set, and we will see who overfit to dev.

**How the arena works**
- Your prompt can use placeholders: `{text}` (the item), `{labels}`, `{codebook}` (the official definitions), `{schema}` (extraction).
- The harness adds one output rule so everyone is parsed the same way. You do **not** need to ask for a specific format.
- Every run costs real money from your allowance and appears on the board. **Write your hypothesis in `note=` before you run.**
- The hidden test set stays with the instructor. Scores you see here are on the *dev* set.

In [ ]:
# ===== Run this cell first =====
import os, sys, hashlib, json
KIT_URL = "https://github.com/atsizelti/prompt_engineering.git"   # public repo with the kit
KIT_DIR = "arena_kit/prompt_arena_kit"                            # folder inside the clone
import importlib.util
if os.path.exists("arena_kit"):
    os.system("git -C arena_kit pull -q")                # always refresh a copy cloned earlier in this session
elif importlib.util.find_spec("arena") is None:
    os.system(f"git clone -q {KIT_URL} arena_kit")
if os.path.exists(KIT_DIR) and KIT_DIR not in sys.path:
    sys.path.insert(0, KIT_DIR)
import arena

NICK      = "your-nickname"      # pick a nickname (no real name needed)
PROVIDER  = "openai"             # "openai", or "mock" (practice with no key)
ENDPOINT  = "https://script.google.com/macros/s/AKfycbxdVYynFuPHaHg73kTFzL5R0TYXrPA8Tr0adMEsb0LKUS5vpDgysktWLPTTcLWrS7Jz/exec"     # the class board (already filled in: you do not need to change it)
TOKEN     = "f-BEqiAfHhAc2O-b"
ALLOWANCE = 0.06                # dollars you may spend (the whole class shares one small budget; re-running this cell does not reset it)

if os.environ.get("ARENA_MOCK"):
    PROVIDER = "mock"
if os.environ.get("ARENA_NO_BOARD"):                  # used by the instructor's offline tests
    ENDPOINT = TOKEN = ""
KEY = None
if PROVIDER != "mock":
    try:
        from google.colab import userdata
        KEY = userdata.get("OPENAI_API_KEY")
    except Exception:
        KEY = os.environ.get("OPENAI_API_KEY")
try:                                                  # the board address and password, set once as Colab secrets
    from google.colab import userdata
    ENDPOINT = ENDPOINT or userdata.get("BOARD_ENDPOINT")
    TOKEN = TOKEN or userdata.get("BOARD_TOKEN")
except Exception:
    ENDPOINT = ENDPOINT or os.environ.get("BOARD_ENDPOINT", "")
    TOKEN = TOKEN or os.environ.get("BOARD_TOKEN", "")
ENDPOINT, TOKEN = "".join(ENDPOINT.split()), "".join(TOKEN.split())   # remove hidden spaces and line breaks from pasting
arena.setup(NICK, provider=PROVIDER, api_key=KEY, endpoint=ENDPOINT or None,
            token=TOKEN or None, allowance_usd=ALLOWANCE)
arena.set_round("R7")

In [ ]:
BEST = """Classify this newspaper snippet by how it frames the dockworkers' strike.
Options: {labels}

Codebook:
{codebook}

Text: {text}"""            # <- paste your best prompt from R2 here
candidates = []
res = arena.evaluate(BEST, task="A", condition="hand", note="R7 start: my best hand-written prompt")
candidates.append((BEST, res)); res.summary()

In [ ]:
# Reflective loop. Each step costs one dev run plus one rewrite call. Stop when your allowance is low.
cur, cur_res = BEST, res
for step in range(3):
    if arena.remaining() < 0.05:
        print("Allowance low: stopping."); break
    new = arena.reflect_and_rewrite(cur, cur_res)
    if new == cur:
        print("No change proposed."); break
    r = arena.evaluate(new, task="A", condition="auto", note=f"R7 reflective step {step+1}")
    candidates.append((new, r))
    print(f"step {step+1}: F1 {r.metrics['macro_f1']:.3f} (was {cur_res.metrics['macro_f1']:.3f}); cost/1k ${r.metrics['cost_per_1k']:.2f}")
    if r.metrics["macro_f1"] >= cur_res.metrics["macro_f1"]:
        cur, cur_res = new, r

In [ ]:
# Pareto set: keep candidates not beaten on both F1 and cost
pts = [(r.metrics["cost_per_1k"], r.metrics["macro_f1"], i) for i, (_, r) in enumerate(candidates)]
front = [i for c, f, i in pts if not any((c2 <= c and f2 >= f) and (c2 < c or f2 > f) for c2, f2, _ in pts)]
for i in front:
    p, r = candidates[i]
    print(f"candidate {i}: F1 {r.metrics['macro_f1']:.3f}  cost/1k ${r.metrics['cost_per_1k']:.2f}")

### Choose and submit
Pick **one** prompt. Remember: dev F1 is optimistic, because you tuned on it.

In [ ]:
CHOICE = front[0]                      # <- change to the index you prefer
final_prompt, final_res = candidates[CHOICE]
print(final_prompt)
arena.submit(final_prompt, task="A", dev_f1=round(final_res.metrics["macro_f1"], 4))

### Keep your work (graded)
Save your final prompt and a few test cases as a small `prompts.py` file: version it, note the model you used, and add 3 test inputs with the label you expect.

In [ ]:
lines = ['"""Final prompt. Model: %s"""' % arena._cfg["model"],
         "PROMPT = " + repr(final_prompt),
         "",
         "TESTS = [",
         '    ("The Whitsun regatta was well attended.", "OFF_TOPIC"),',
         "    # add two more (text, expected_label) pairs",
         "]"]
open("prompts.py", "w").write("\n".join(lines) + "\n")
print("saved prompts.py")